# State Migration - 01: When the schema changes underneath you

> **MLCourse - Agentic AI - LangGraph - Module 10**

[`03_persistence_checkpointing`](../03_persistence_checkpointing/README.md)
taught you to save graph state to a checkpointer so a conversation survives a
restart. It quietly assumed something that stops being true the moment you
ship a second version of your agent:

> **that the code reading a checkpoint is the same code that wrote it.**

It is not. Checkpoints are **data at rest**, and data at rest outlives the
code that produced it. A thread paused for human approval on Friday gets
resumed on Monday - by a deployment that shipped over the weekend, with a
`TypedDict` that has a new field in it.

This module is about that gap. It is the database-migration problem, arriving
in a place most people do not expect to find it.

### What you will learn in this module

1. What a checkpoint physically contains, and why nothing validates it.
2. The two ways a schema change breaks a live thread - reproduced, not
   described.
3. Which changes are **additive** (safe) and which are **breaking**.
4. How to version your state and write a migration function.
5. How to recover checkpoints written by an older schema - lazily and in bulk.

### No API key needed

Every node in this module is an ordinary Python function returning a dict.
There are **no LLM calls anywhere in module 10** - the subject is state
plumbing, and a model would only add noise and cost. Nodes are deliberately
boring so the checkpoint behaviour is the only thing moving.

### Setup


In [ ]:
# Nothing here needs a key. We use SqliteSaver rather than InMemorySaver
# because the whole point of this module is state that OUTLIVES a process --
# an in-memory checkpointer would hide the problem we are studying.

import json
import os
import sqlite3
from typing import TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

DB_PATH = "migration_demo.db"

# Start from a clean database so this notebook is reproducible when re-run.
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)

print("Module 10: State Migration and Versioning")
print(f"checkpoint store: {DB_PATH} (fresh)")
print("no API key required -- there are no LLM calls in this module")


### 1. Version 1 of the agent - the code that is already in production

Imagine a small order-processing agent that shipped three months ago. Two
fields of state, two nodes. It has been running happily and has written
thousands of checkpoints.

### V1: the agent that is already deployed


In [ ]:
class OrderStateV1(TypedDict):
    """Version 1 of the state. Note what is NOT here: no priority, no notes."""
    order_id: str
    status: str


def validate_v1(state: OrderStateV1) -> dict:
    """Pretend to validate the order."""
    return {"status": "validated"}


def process_v1(state: OrderStateV1) -> dict:
    """Pretend to process it."""
    return {"status": f"processed:{state['order_id']}"}


builder_v1 = StateGraph(OrderStateV1)
builder_v1.add_node("validate", validate_v1)
builder_v1.add_node("process", process_v1)
builder_v1.add_edge(START, "validate")
builder_v1.add_edge("validate", "process")
builder_v1.add_edge("process", END)

agent_v1 = builder_v1.compile(checkpointer=checkpointer)

# Three customers have used the v1 agent. Each gets its own thread.
for i in range(3):
    cfg = {"configurable": {"thread_id": f"order-{i}"}}
    result = agent_v1.invoke({"order_id": f"A-10{i}", "status": "new"}, cfg)
    print(f"thread order-{i}: {result}")

print("\n3 threads now have persisted state written by V1.")


### 2. What is actually in a checkpoint?

Before breaking anything, look at what was stored. This is the part that makes
the rest of the module obvious: a checkpoint is **a plain dict of channel
values**. There is no schema attached to it, no version stamp, and no
validation on the way back in.

### Looking inside a checkpoint


In [ ]:
cfg0 = {"configurable": {"thread_id": "order-0"}}
tuples = list(checkpointer.list(cfg0))

print(f"thread 'order-0' has {len(tuples)} checkpoints (one per superstep)\n")

latest = tuples[0]          # list() returns newest first
print("keys stored in the checkpoint object:")
print(f"  {list(latest.checkpoint.keys())}\n")
print("channel_values -- this IS your state:")
print(f"  {latest.checkpoint['channel_values']}\n")
print("metadata:")
print(f"  {latest.metadata}\n")

print("Note what is ABSENT:")
print("  * no schema")
print("  * no version number")
print("  * nothing that says 'this was written by OrderStateV1'")


### The single most important sentence in this module

> **A checkpoint is an untyped dict. `TypedDict` is a *static* annotation -
> it disappears at runtime and validates nothing.**

`OrderStateV1` was never written to the database, and LangGraph never checks a
loaded checkpoint against the state class of the graph loading it. The state
class tells LangGraph which *channels* exist and how to merge updates into
them; it is not a contract enforced against stored data.

That design is what makes checkpointing fast and flexible. It is also what
makes the next section possible.

### 3. Version 2 ships - and the old threads break

Three months later, a feature request: orders need a **priority**. So the
state class grows a field, and a node starts reading it.

This is the most ordinary change imaginable. Let's see what it does to the
threads that are already out there.

### V2: one new field, and a node that reads it


In [ ]:
class OrderStateV2(TypedDict):
    order_id: str
    status: str
    priority: str          # <-- NEW in v2


def validate_v2(state: OrderStateV2) -> dict:
    return {"status": "validated"}


def process_v2(state: OrderStateV2) -> dict:
    # The innocuous line that breaks everything. On a fresh thread this is
    # fine. On a thread whose state was written by V1, 'priority' is simply
    # not there.
    return {"status": f"processed:{state['order_id']}:{state['priority']}"}


builder_v2 = StateGraph(OrderStateV2)
builder_v2.add_node("validate", validate_v2)
builder_v2.add_node("process", process_v2)
builder_v2.add_edge(START, "validate")
builder_v2.add_edge("validate", "process")
builder_v2.add_edge("process", END)

agent_v2 = builder_v2.compile(checkpointer=checkpointer)

# A brand-new customer arrives after the deploy. Everything is fine.
new_cfg = {"configurable": {"thread_id": "order-new"}}
print("NEW thread on V2:")
print(" ", agent_v2.invoke(
    {"order_id": "A-200", "status": "new", "priority": "high"}, new_cfg))


### Failure mode 1: an existing thread takes its next turn


In [ ]:
# The customer from thread order-0 comes back. Their state is loaded from the
# checkpoint -- written by V1, so it has no 'priority' channel at all.

print("EXISTING thread order-0, now running on V2:")
try:
    result = agent_v2.invoke({"order_id": "A-100"}, cfg0)
    print("  succeeded:", result)
except KeyError as e:
    print(f"  KeyError: {e}")
    print("\n  The node asked for state['priority'].")
    print("  The checkpoint was written before that field existed.")
    print("  Nothing warned us -- not at deploy time, not at load time.")


### Failure mode 1: the next turn

A `KeyError` in production, on a code path that was tested and works
perfectly - for every thread created *after* the deploy.

This is what makes the bug nasty:

- **Your tests pass.** Tests create fresh threads, which get the new schema.
- **Your smoke test passes.** You try it yourself after deploying; you are a
  new thread.
- **It fails only for existing users**, which is the population you least want
  to break, and it fails in proportion to how successful your product is.

The failure is also *silent until triggered*. The bad checkpoints sat in the
database the whole time. Nothing scanned them; nothing could.

### Failure mode 2: a thread suspended mid-execution


In [ ]:
# This one is worse. A thread paused at a human-approval interrupt is holding
# a checkpoint AND a pending task. Deploy a schema change while it waits, and
# the resume itself fails -- there is no "start over" to fall back on.

from langgraph.types import Command, interrupt


class ReviewStateV1(TypedDict):
    document: str
    decision: str


def request_review_v1(state: ReviewStateV1) -> dict:
    answer = interrupt({"question": f"Approve {state['document']}?"})
    return {"decision": answer}


rb1 = StateGraph(ReviewStateV1)
rb1.add_node("review", request_review_v1)
rb1.add_edge(START, "review")
rb1.add_edge("review", END)
review_v1 = rb1.compile(checkpointer=checkpointer)

review_cfg = {"configurable": {"thread_id": "review-1"}}
paused = review_v1.invoke({"document": "contract.pdf", "decision": ""}, review_cfg)
print("FRIDAY -- thread suspended waiting for a human:")
print(f"  state now: {review_v1.get_state(review_cfg).values}")
print(f"  interrupted: {bool(paused.get('__interrupt__'))}")


### ... over the weekend, v2 of the review agent ships ...


In [ ]:
class ReviewStateV2(TypedDict):
    document: str
    decision: str
    reviewer: str          # <-- NEW: we now record WHO approved


def request_review_v2(state: ReviewStateV2) -> dict:
    answer = interrupt({"question": f"Approve {state['document']}?"})
    return {"decision": answer, "reviewer": state["reviewer"]}


rb2 = StateGraph(ReviewStateV2)
rb2.add_node("review", request_review_v2)
rb2.add_edge(START, "review")
rb2.add_edge("review", END)
review_v2 = rb2.compile(checkpointer=checkpointer)

print("MONDAY -- the human approves. Resume under the NEW schema:")
try:
    print("  ", review_v2.invoke(Command(resume="approved"), review_cfg))
except KeyError as e:
    print(f"  KeyError: {e}")
    print("\n  The human's approval is now unreachable. The thread cannot")
    print("  move forward (the node crashes) and cannot be restarted (that")
    print("  would discard the approval and ask the human again).")
    print("  This thread is WEDGED until someone migrates its state.")


### Failure mode 2: the suspended thread

This is the one that actually hurts, and it is specific to agents in a way the
classic database-migration problem is not.

A suspended thread is **work in progress**. It holds:

- accumulated state from before the pause,
- a pending task that knows where to resume,
- and, in a human-in-the-loop graph, a decision a real person already made.

"Just restart the thread" is not available. Restarting throws away the
approval and asks the human again - and in a workflow where the interrupt
represents a payment authorisation or a legal sign-off, asking again is not a
minor inconvenience.

The longer a thread can stay suspended, the wider your exposure. A graph that
waits days for human approval - see
[`04_human_in_the_loop`](../04_human_in_the_loop/README.md), and
`06_agent_patterns/14_async_human_approval` for the asynchronous version -
has days of deploys to survive.

### 4. Why LangGraph does not just solve this for you

A reasonable reaction is: *why doesn't the framework detect this?*

It cannot, and the reason is worth understanding rather than resenting.

**LangGraph does not know what your fields mean.** Suppose it noticed that a
checkpoint lacks `priority` while the current schema has it. What should it
do?

- Insert a default? It has no idea what a sensible default is, and a wrong
  default is worse than a crash - it silently processes an urgent order as
  routine.
- Refuse to load the thread? That turns a bug affecting one code path into a
  total outage for every existing user.
- Drop the thread? Obviously not.

Every automatic policy is wrong for some application. So LangGraph does the
only defensible thing: it loads exactly what was stored and lets **your** code
decide, because your code is the only thing that knows a missing `priority`
should mean `"normal"`.

The framework's job is to persist your state faithfully. Deciding what an old
state *means* under new rules is a domain question, and that is exactly what a
migration function is.

**The practical consequence:** if you use a checkpointer, schema migration is
your responsibility, and it starts the moment you have one deployed version.

### Key takeaways

- A checkpoint is an **untyped dict of channel values**. `TypedDict` is a
  static annotation that validates nothing at runtime, and nothing records
  which schema wrote a given checkpoint.
- Checkpoints **outlive the code that wrote them**. Any deploy that changes
  the state schema is a data migration, whether or not you treat it as one.
- **Failure mode 1** - an existing thread takes its next turn and a node
  raises `KeyError` on a field that did not exist when the thread started.
  Tests and smoke tests miss this, because they create fresh threads.
- **Failure mode 2** - a thread suspended at an `interrupt()` cannot be
  resumed *or* safely restarted, because restarting discards a decision a
  human already made. The thread is wedged.
- LangGraph **cannot** fix this automatically: only your code knows what a
  missing field should default to.

**Next:** `02_additive_vs_breaking_changes.ipynb` - a taxonomy of which schema
changes are safe, which are not, and which are dangerous precisely because
they *do not* raise an error.